# Funktionen für Parsen, Entitäten und Namespaces


Die Datei lb-test.xml ist eine einfache heiEditions-XML-Datei. Da werden Entites verwendet, die in https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/declarations/heieditions-entities.txt" definiert sind. Im nächsten Code-Abschnitt kann man den Inhalt der Datei sehen.

In [ ]:
example_file_path = 'beispiele/beispiel_data/lb-test.xml'
import codecs
with codecs.open(example_file_path, 'r', 'utf-8') as example_file:
    example = example_file.read()
    print(example)

Wenn wir so eine Datei mit lxml/etree parsen wollen, kommt eine Fehlermeldung, da die externe Entities nicht geladen werden können.

In [ ]:
from lxml import etree as et

try:
    tree = et.parse('beispiele/beispiel_data/lb-test.xml')
except SyntaxError as e:
    print(e)
    pass

heipy bietet ein eigenes Parser, das diese Datei parsen kann.

In [ ]:
from heipy.parsers import HeiEditionsParser

heiparser = HeiEditionsParser()
tree = et.parse('beispiele/beispiel_data/lb-test.xml', parser= heiparser)
root = tree.getroot()
print(root)

Wenn wir in einer TEI solchen Datei XPath verwenden wollen, müssen wir entweder die Namespaces in geschweiften Klammern schreiben oder die Präfixe definieren. Also:

In [4]:
facsimile = root.findall('.//{http://www.tei-c.org/ns/1.0}facsimile')
facsimile = root.findall('.//tei:facsimile', namespaces= {'tei':'http://www.tei-c.org/ns/1.0'})

Die wichtigsten Präfixe (tei,xml,hei,hc,page,mets) werden in heipy schon in einer Variabel definiert, die importiert werden kann.

In [ ]:
from heipy.namespaces import ns 

facsimile = root.findall('.//tei:facsimile', namespaces= ns)
print(facsimile)

Manchmal müssen wir mit lxml auch diese Präfixe vor dem Elementname schreiben. Das können wir auch mit der Funktion `prefix_format` aus heipy machen. Hier ein Beispiel, wenn wir ein neues `<link>` Element in TEI-Namespace mit etree erzeugen wollen oder alle xml:id von `<l>` Elemente suchen.

In [ ]:
from heipy.namespaces import prefix_format

new_elelement = et.Element(prefix_format('tei','link'))

for line in root.findall('.//tei:l', namespaces=ns):
    line_id = line.get(prefix_format('xml','id'))
    print(line_id)

# Transformations-Pipeline

Eine Pipeline besteht aus eine Serie von Schritten, die nacheinander ausgeführt werden. Es gibt unterschiedliche Arten von Schritten: XsltStep, PythonStep, AddAttribute, ValidationStep, DeleteStep. Diese Klassen sind in heipy.heipipe.steps definiert.

In [ ]:
from heipy.heipipe.steps import Pipeline, XsltStep

pipe = Pipeline(name='Example_Pipe')

schritt1 = XsltStep(['src/heipy/heipipe/xslt/text_initials.xsl'], name="Initials")
pipe.add_step(schritt1)

schritt2 = XsltStep(['src/heipy/heipipe/xslt/text_markNoteAsEditorial.xsl'], name="Mark_note_as_editorial",
                    parameters= [{'note_classes': "hc:Comment"}])
pipe.add_step(schritt2)

result = pipe.execute(example_file_path)



Es gibt unterschiedliche Arten von Schritten: XsltStep, AddAttribute, DeleteStep, UnwrapStep, ValidationStep

In [ ]:
from heipy.heipipe.steps import PythonStep, AddAttribute, ValidationStep, DeleteStep, UnwrapStep

pipe.add_step( DeleteStep(['facsimile']) )
# help(DeleteStep)

# pipe.add_step(ValidationStep())
# help(ValidationStep)

pipe.add_step(AddAttribute('//tei:title', 'ana', 'hc:MainTitle'))
# help(AddAttribute)

pipe.add_step(UnwrapStep([{'element_name': 'w'}]))
# help(UnwrapStep)


A PythonStep is the most complex kind of step. It requires a function that takes as a parameter a root from an xml object element and a parameters argument that can be empty. It must return the edited root. For example:

In [ ]:
def add_ptr_after_l(root, parameters):
    ls = root.findall('.//tei:l', namespaces=ns)
    for l in ls:
        et.SubElement(l, prefix_format('tei','ptr'), nsmap=ns)
    return root

pipe.add_step(PythonStep(add_ptr_after_l))

pipe.execute(example_file_path)


Da die meisten Pipelines immer sehr ähnlich sind, werden default-Pipelines in heipy definiert. Zur Zeit gibt es semantic und sourceDoc

In [10]:
from heipy.heipipe.pipeline_library.sourcedoc import SourceDocPipe
from heipy.heipipe.pipeline_library.semantic import SemanticPipe

semantic_pipe = SemanticPipe()
sourcedoc_pipe = SourceDocPipe()

Die einzelnen Schritten der Default-Pipelines werden in heipy.heipipe.step_library definiert. Man kann diese mit dem get_steps() Funktion auflisten.

In [ ]:
for i, step in enumerate(sourcedoc_pipe.get_steps()):
    print(f'{i}: {step}')

Man kann neue Schritte hinzufügen mit add_step(). Per Default am Ende der Pipeline, aber mit möglich index (s. oben um die Indexes zu sehen). Alternativ kann man after_step() oder before_step() verwenden

In [ ]:
sourcedoc_pipe = SourceDocPipe()

sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], name="new_step_1"))

sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], name="new_step_2"), 
                        at_index= 0)
sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], name="new_step_3"), 
                        after_step='Whitespaces')
sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], name="new_step_4"), 
                        before_step='new_step_2')

sourcedoc_pipe.set_pipestep_parameter('mark_note_as_editorial','note_classes', 'hc:Comment')

for i, step in enumerate(sourcedoc_pipe.get_steps()):
    print(f'{i}: {step}')

